# Vector Search: Chunked Exact Top-k, an IVF Index and Recall@k — Solution

**Difficulty**: 🟡 Medium

### Problem Statement

Image–text retrieval with CLIP/SigLIP, retrieval-augmented generation, near-duplicate removal in pre-training data and recommendation all reduce to the same operation: **embed everything, then find the nearest neighbours of a query embedding**. Exact search is one matrix multiply plus a top-k. At scale, though, you cannot materialise the $Q \times N$ score matrix ($10^4$ queries × $10^8$ images in float32 is 4 TB), so you **stream the database in chunks** and keep a running top-k. Once even a full scan is too slow, you switch to an **approximate index**. The simplest one, and the workhorse of FAISS, is the **inverted file (IVF)**:

1. **Train** a coarse quantizer: k-means with `nlist` centroids.
2. **Add**: assign each database vector to its nearest centroid and append its id to that centroid's *inverted list*.
3. **Search**: find the `nprobe` centroids nearest to the query and score exactly only the vectors in those lists. A search scans about `nprobe / nlist` of the database, and `nprobe` trades recall for speed.

Approximate search quality is measured by **recall@k**: the fraction of the true top-k (from exact search) that the index returns.

**Metrics.** For a query $q$ and database vector $d$:

| metric | score | better | notes |
|---|---|---|---|
| `"cosine"` | $\dfrac{q^\top d}{\lVert q\rVert\,\lVert d\rVert}$ | higher | = dot product of L2-normalised vectors |
| `"dot"` | $q^\top d$ | higher | maximum inner product search |
| `"l2"` | $\lVert q - d\rVert^2 = \lVert q\rVert^2 - 2\,q^\top d + \lVert d\rVert^2$ | **lower** | squared distance via the expansion, clamped at 0 |

---

### Requirements

1. **`exact_topk(queries, database, k, metric="cosine", chunk_size=1024) -> (scores, indices)`**
   - `queries`: `(Q, D)`, `database`: `(N, D)`; returns `scores` `(Q, k)` (float) and `indices` `(Q, k)` (int64 database row numbers), best first. For `"l2"` that means the **k smallest** squared distances, in ascending order.
   - `"l2"` uses the expansion above (one GEMM per chunk), **clamped at 0**: in float32 the expansion can come out slightly negative.
   - Process the database in chunks of `chunk_size` rows and **never materialise the full `(Q, N)` score matrix**. Keep a running `(Q, k)` best-so-far and merge each chunk into it.
   - Must work for any `chunk_size ≥ 1`, including `chunk_size < k` and `N` not divisible by `chunk_size`. You may assume `1 ≤ k ≤ N`.
2. **`IVFIndex(nlist, nprobe, metric="cosine", n_iter=20, seed=0)`**
   - `train(vectors)`: k-means with `nlist` centroids. Initialise with `nlist` distinct random training rows chosen with a `torch.Generator` seeded by `seed`, then run `n_iter` Lloyd iterations (assign every vector to its nearest centroid by squared L2, then move each centroid to the mean of its cluster; a centroid whose cluster is empty stays where it is). For `metric="cosine"`, L2-normalise the vectors first. Store the result in `self.centroids` `(nlist, D)`.
   - `add(vectors)`: (normalise for cosine,) assign each vector to its nearest centroid and append its **global id** to that centroid's list. Ids are row numbers in insertion order and **continue across `add` calls** (the second call's first vector gets id = number already added).
   - `search(queries, k) -> (scores, ids)`, both `(Q, k)`: probe the `self.nprobe` centroids nearest to each query (read `self.nprobe` at search time, since callers change it between searches), score every candidate in those lists **exactly** with the index's metric (same conventions as `exact_topk`), and return the best `k`, best first. If a query has fewer than `k` candidates, pad the end of its row with id **`-1`** and score **`-inf`** (`"cosine"`, `"dot"`) or **`+inf`** (`"l2"`), i.e. the worst possible score.
3. **`recall_at_k(approx_indices, exact_indices) -> float`**: both `(Q, k)`. Return the mean over queries of $|\mathrm{approx}_i \cap \mathrm{exact}_i| / k$. Order does not matter, and `-1` never counts as a hit.

---

### Constraints

- ✅ Pure PyTorch. Write the k-means yourself (no sklearn / faiss).
- ✅ Vectorise everything that touches all vectors at once: k-means assignment (one distance matrix + `argmin`), the centroid update (`index_add_`, `scatter_add_` or a one-hot matmul), chunk scoring and `recall_at_k`.
- ✅ A Python loop over **queries** inside `IVFIndex.search` is fine. Candidate lists have different lengths, and real IVF kernels also process one query at a time. A loop over chunks in `exact_topk` is the point of the exercise.
- ❌ No `torch.cdist` in `exact_topk`: write the expansion (it is what makes l2 search a single GEMM).
- ❌ No Python loop over individual database vectors.

---

<details>
  <summary>💡 Hint</summary>

  - If a vector is in the global top-k, it is also in the top-k of whichever chunk contains it. So what do you need to keep from each chunk, and how do you merge it with what you already have?
  - `torch.topk` returns *positions within the tensor you gave it*. What has to be added to turn a position inside a chunk into a database row number? And what happens if you ask a 7-row chunk for its top 10?
  - For l2, which `topk` argument picks the smallest values?
  - k-means update: how can you sum all vectors of each cluster in one call, without a loop over clusters?
  - IVF with `nprobe = nlist` scans every list, so it must give exactly the exact-search answer. That makes a good first test.
  - For recall, broadcasting `(Q, k, 1) == (Q, 1, k)` compares every returned id with every true id.

</details>

---

In [ ]:
import time

import torch
import torch.nn.functional as F

In [ ]:
# Synthetic "CLIP-like" embeddings: 4000 database vectors in 64-d drawn around 100 broad,
# overlapping semantic clusters (think: image embeddings) and 100 queries from the same
# distribution (think: captions). A separate sample is the index's training set.
torch.manual_seed(0)
D, N_DB, N_Q, N_CLUSTERS, NOISE = 64, 4000, 100, 100, 1.5
centers = torch.randn(N_CLUSTERS, D)
database = centers[torch.randint(0, N_CLUSTERS, (N_DB,))] + NOISE * torch.randn(N_DB, D)
queries = centers[torch.randint(0, N_CLUSTERS, (N_Q,))] + NOISE * torch.randn(N_Q, D)
train_set = centers[torch.randint(0, N_CLUSTERS, (N_DB,))] + NOISE * torch.randn(N_DB, D)


def brute_force(q, db, k, metric):
    """The obvious, memory-hungry reference: full (Q, N) score matrix + full sort."""
    if metric == "cosine":
        q, db = F.normalize(q, dim=-1), F.normalize(db, dim=-1)
    s = torch.cdist(q, db) ** 2 if metric == "l2" else q @ db.T
    order = s.argsort(dim=1, descending=(metric != "l2"))[:, :k]
    return s.gather(1, order), order


def assert_same_topk(got, ref, what):
    (s, i), (rs, ri) = got, ref
    assert s.shape == rs.shape and i.shape == ri.shape, \
        f"{what}: got shapes {tuple(s.shape)} / {tuple(i.shape)}, expected {tuple(rs.shape)}"
    assert torch.allclose(s, rs, rtol=1e-4, atol=1e-3), \
        f"{what}: scores differ from brute force (max diff {(s - rs).abs().max():.2e})"
    mismatch = (i != ri).float().mean().item()       # tolerate rare float near-ties
    assert mismatch < 0.01, f"{what}: {mismatch:.1%} of the returned ids differ from brute force"


print("database", tuple(database.shape), " queries", tuple(queries.shape))

In [ ]:
METRICS = ("cosine", "dot", "l2")


def exact_topk(queries: torch.Tensor, database: torch.Tensor, k: int,
               metric: str = "cosine", chunk_size: int = 1024):
    """
    Exact k-nearest-neighbour search, streaming the database in chunks.

    Args:
        queries:    (Q, D) query embeddings.
        database:   (N, D) database embeddings.
        k:          number of neighbours, 1 <= k <= N.
        metric:     "cosine" | "dot" (higher is better) or "l2" (squared distance, lower is better).
        chunk_size: database rows scored at a time; never build the full (Q, N) matrix.

    Returns:
        scores:  (Q, k) float, best first (descending for cosine/dot, ascending for l2).
        indices: (Q, k) int64 database row numbers.
    """
    if metric not in METRICS:
        raise ValueError(f"metric must be one of {METRICS}, got {metric!r}")
    n = database.shape[0]
    if not 1 <= k <= n:
        raise ValueError(f"need 1 <= k <= {n}, got k={k}")
    largest = metric != "l2"                                   # l2: smaller is better
    if metric == "cosine":                                     # cosine = dot of unit vectors
        queries = F.normalize(queries, dim=-1)                 # (an index stores these once)
        database = F.normalize(database, dim=-1)
    q_sq = (queries * queries).sum(dim=1, keepdim=True)        # (Q, 1), used by l2

    best_s = best_i = None
    for start in range(0, n, chunk_size):
        chunk = database[start:start + chunk_size]             # (c, D), the last one may be short
        s = queries @ chunk.T                                  # (Q, c): the only GEMM
        if metric == "l2":                                     # ||q||^2 - 2 q.d + ||d||^2 >= 0
            s = (q_sq - 2 * s + (chunk * chunk).sum(dim=1)).clamp_min(0)
        top = s.topk(min(k, s.shape[1]), dim=1, largest=largest)    # a chunk may hold < k rows
        cand_s, cand_i = top.values, top.indices + start            # chunk position -> row id
        if best_s is not None:                                      # merge with the running best
            cand_s = torch.cat([best_s, cand_s], dim=1)             # (Q, <= 2k)
            cand_i = torch.cat([best_i, cand_i], dim=1)
        keep = cand_s.topk(min(k, cand_s.shape[1]), dim=1, largest=largest)
        best_s, best_i = keep.values, cand_i.gather(1, keep.indices)
    return best_s, best_i

In [ ]:
for metric in ("cosine", "dot", "l2"):
    ref = brute_force(queries, database, 10, metric)
    for cs in (7, 100, 999, 4000, 10_000):          # chunks smaller than k, ragged last chunk, one chunk
        got = exact_topk(queries, database, 10, metric=metric, chunk_size=cs)
        assert_same_topk(got, ref, f"exact_topk({metric}, chunk_size={cs})")

# l2: every database vector is its own nearest neighbour, at distance ~0 and never below 0.
s, i = exact_topk(database[:5].clone(), database, 3, metric="l2", chunk_size=512)
assert i[:, 0].tolist() == [0, 1, 2, 3, 4], i[:, 0].tolist()
assert (s >= 0).all() and s[:, 0].max() < 1e-3, s[:, 0]

t = time.perf_counter()
exact_topk(queries, database, 10, metric="cosine", chunk_size=1024)
print(f"exact search over {N_DB} vectors: {1e3 * (time.perf_counter() - t):.1f} ms")
print("exact_topk tests passed")

In [ ]:
def recall_at_k(approx_indices: torch.Tensor, exact_indices: torch.Tensor) -> float:
    """
    Args:
        approx_indices: (Q, k) ids from an approximate search (-1 = padding).
        exact_indices:  (Q, k) ids from exact search.

    Returns:
        mean over queries of |approx_i ∩ exact_i| / k  (order-insensitive; -1 never counts).
    """
    k = exact_indices.shape[1]
    # (Q, k', 1) == (Q, 1, k): is each returned id anywhere in the true top-k?
    hit = (approx_indices.unsqueeze(2) == exact_indices.unsqueeze(1)).any(dim=2)   # (Q, k')
    hit &= approx_indices >= 0                                                     # padding never hits
    return (hit.sum(dim=1).float() / k).mean().item()

In [ ]:
assert recall_at_k(torch.tensor([[1, 2, 3]]), torch.tensor([[1, 2, 3]])) == 1.0
assert recall_at_k(torch.tensor([[3, 1, 2]]), torch.tensor([[1, 2, 3]])) == 1.0, "order must not matter"
assert abs(recall_at_k(torch.tensor([[1, -1, -1]]), torch.tensor([[1, 2, 3]])) - 1 / 3) < 1e-6, \
    "padding (-1) is not a hit, and the denominator is k"
assert abs(recall_at_k(torch.tensor([[1, 2], [5, 6]]), torch.tensor([[2, 9], [7, 8]])) - 0.25) < 1e-6, \
    "average over queries: (1/2 + 0/2) / 2"
print("recall_at_k tests passed")

In [ ]:
class IVFIndex:
    """Inverted-file index: a k-means coarse quantizer plus one list of ids per centroid."""

    def __init__(self, nlist: int, nprobe: int, metric: str = "cosine",
                 n_iter: int = 20, seed: int = 0):
        if metric not in METRICS:
            raise ValueError(f"metric must be one of {METRICS}, got {metric!r}")
        self.nlist, self.nprobe, self.metric = nlist, nprobe, metric
        self.n_iter, self.seed = n_iter, seed
        self.centroids = None                                   # (nlist, D) after train()
        self.vectors = None                                     # (ntotal, D) as stored
        self.lists = [torch.empty(0, dtype=torch.long) for _ in range(nlist)]

    def _prep(self, x):
        return F.normalize(x, dim=-1) if self.metric == "cosine" else x

    def _nearest_centroids(self, x, n):
        """(M, n) ids of the n nearest centroids (squared L2) for every row of x."""
        d = ((x * x).sum(1, keepdim=True) - 2 * x @ self.centroids.T
             + (self.centroids * self.centroids).sum(1))                     # (M, nlist)
        return d.topk(n, dim=1, largest=False).indices

    def train(self, vectors: torch.Tensor):
        """k-means on (N, D) training vectors -> self.centroids (nlist, D)."""
        x = self._prep(vectors)
        if x.shape[0] < self.nlist:
            raise ValueError(f"need at least nlist={self.nlist} training vectors")
        g = torch.Generator().manual_seed(self.seed)
        self.centroids = x[torch.randperm(x.shape[0], generator=g)[: self.nlist]].clone()
        for _ in range(self.n_iter):
            assign = self._nearest_centroids(x, 1).squeeze(1)                 # (N,)
            onehot = F.one_hot(assign, self.nlist).to(x.dtype)                # (N, nlist)
            sums = onehot.T @ x                                               # (nlist, D) cluster sums
            counts = onehot.sum(0).unsqueeze(1)                               # (nlist, 1)
            # an empty cluster keeps its previous centroid instead of becoming 0/0 = NaN
            self.centroids = torch.where(counts > 0, sums / counts.clamp_min(1), self.centroids)
        return self

    def add(self, vectors: torch.Tensor):
        """Store (M, D) vectors; ids continue from the number of vectors already added."""
        if self.centroids is None:
            raise RuntimeError("call train() before add()")
        x = self._prep(vectors)
        start = 0 if self.vectors is None else self.vectors.shape[0]
        ids = torch.arange(start, start + x.shape[0])                         # global ids
        assign = self._nearest_centroids(x, 1).squeeze(1)
        self.vectors = x if self.vectors is None else torch.cat([self.vectors, x])
        for c in assign.unique().tolist():                                    # loop over lists
            self.lists[c] = torch.cat([self.lists[c], ids[assign == c]])
        return self

    def search(self, queries: torch.Tensor, k: int):
        """
        Returns (scores, ids), both (Q, k): exact scores of the best candidates found in the
        self.nprobe nearest lists, best first; missing results padded at the end with id -1
        and score -inf (cosine/dot) or +inf (l2).
        """
        q = self._prep(queries)
        largest = self.metric != "l2"
        worst = float("-inf") if largest else float("inf")
        scores = torch.full((q.shape[0], k), worst, dtype=q.dtype)
        ids = torch.full((q.shape[0], k), -1, dtype=torch.long)
        if self.vectors is None:
            return scores, ids
        probes = self._nearest_centroids(q, min(self.nprobe, self.nlist))      # (Q, nprobe)
        for i in range(q.shape[0]):                                           # one query at a time
            cand = torch.cat([self.lists[c] for c in probes[i].tolist()])     # (m,) global ids
            if cand.numel() == 0:
                continue
            vecs = self.vectors[cand]                                         # (m, D)
            s = vecs @ q[i]
            if self.metric == "l2":
                s = ((q[i] * q[i]).sum() - 2 * s + (vecs * vecs).sum(1)).clamp_min(0)
            top = s.topk(min(k, cand.numel()), largest=largest)
            scores[i, : top.values.numel()] = top.values
            ids[i, : top.values.numel()] = cand[top.indices]
        return scores, ids

In [ ]:
k = 10
for metric in ("cosine", "l2"):
    index = IVFIndex(nlist=64, nprobe=64, metric=metric)
    index.train(train_set)
    index.add(database[:2500])
    index.add(database[2500:])                   # ids must continue across add() calls
    exact = brute_force(queries, database, k, metric)
    assert_same_topk(index.search(queries, k), exact, f"IVF({metric}) with nprobe = nlist")

    print(f"metric={metric}:  nprobe  recall@{k}  ms/query")
    recalls = []
    for p in (1, 2, 4, 8, 16, 32, 64):
        index.nprobe = p
        t = time.perf_counter()
        _, ids = index.search(queries, k)
        ms = 1e3 * (time.perf_counter() - t) / N_Q
        recalls.append(recall_at_k(ids, exact[1]))
        print(f"{p:20d}  {recalls[-1]:9.3f}  {ms:8.3f}")
    assert all(b >= a for a, b in zip(recalls, recalls[1:])), "recall must not fall as nprobe grows"
    assert recalls[-1] == 1.0 and recalls[0] > 0.3, recalls

# Fewer candidates than k: pad with id -1 and the worst score.
tiny = IVFIndex(nlist=2, nprobe=2, metric="l2")
tiny.train(database[:6])
tiny.add(database[:6])
s, i = tiny.search(queries[:3], 10)
assert s.shape == (3, 10) and (i[:, :6] >= 0).all(), "the 6 real results come first"
assert (i[:, 6:] == -1).all() and (s[:, 6:] == float("inf")).all(), "pad with id -1 and score +inf (l2)"
print("IVFIndex tests passed")

In [ ]:
# Auto-grade (one-time setup from the repo root: pip install -e python/)
from torchleet import check
check("vector-search", exact_topk=exact_topk, IVFIndex=IVFIndex, recall_at_k=recall_at_k)

### Why it works

- **Chunked top-k.** If a vector is in the global top-k, then fewer than $k$ vectors beat it anywhere, so fewer than $k$ beat it inside its own chunk, and it is in that chunk's top-$\min(k, c)$. The global top-k is therefore contained in the union of the per-chunk top-k sets. Merging each chunk into a running buffer keeps the invariant "the buffer holds the top-k of everything seen so far". Peak memory is $O(Q\,(k + c))$ instead of $O(QN)$, and the result is exact.
- **The l2 expansion.** $\lVert q - d\rVert^2 = \lVert q\rVert^2 - 2q^\top d + \lVert d\rVert^2$ turns all pairwise distances into one GEMM plus two norm vectors (which an index precomputes once). The price is **catastrophic cancellation**: when $\lVert q\rVert^2$ and $\lVert d\rVert^2$ are large and $q \approx d$, the float32 result can be slightly negative, hence the clamp. When exact small distances matter, re-rank the survivors with $(q - d)^2$ computed directly.
- **Cosine through the same machinery.** After L2-normalisation, cosine is the dot product, and for unit vectors $\lVert q - d\rVert^2 = 2 - 2\cos(q, d)$. One normalised index serves both, and normalising once at indexing time is free at query time.
- **IVF.** k-means puts nearby vectors in the same Voronoi cell, so a query's nearest neighbours are very likely in the cells of its nearest centroids. With `nprobe = nlist` every list is scanned, so the answer is exact. Probing the `nprobe` *nearest* centroids makes the scanned sets nested as `nprobe` grows, so recall can only go up. Errors come from neighbours that sit just across a cell boundary.

### Complexity

| | time | extra memory |
|---|---|---|
| `exact_topk` | $O(QND)$ | $O(Q(k + c))$ |
| IVF `train` | $O(I \cdot N \cdot \text{nlist} \cdot D)$, $I$ = `n_iter` | $O(N \cdot \text{nlist})$ for the assignment |
| IVF `search` | $O(Q \cdot \text{nlist} \cdot D) + O(Q \cdot \text{nprobe} \cdot \tfrac{N}{\text{nlist}} \cdot D)$ | $O(\text{nprobe} \cdot \tfrac{N}{\text{nlist}})$ per query |

With $\text{nlist} \approx \sqrt N$ both search terms are $O(\sqrt N)$ per query instead of $O(N)$.

### Common bugs

- Forgetting the chunk offset: ids come back as positions inside the chunk.
- `topk(k)` on a chunk with fewer than `k` rows, which crashes on small or ragged last chunks.
- l2 with `largest=True` (returns the *farthest* vectors), or no clamp (negative "distances", NaN once you take a square root).
- Building the full `(Q, N)` matrix anyway: the broadcast `q_sq + d_sq` without chunking, or collecting every chunk's scores and concatenating them.
- Cosine: normalising the queries but not the stored vectors, or normalising along the wrong dim.
- IVF: ids restarting at 0 on every `add()`; padding with id 0 (a real vector!) instead of -1; empty k-means clusters producing NaN centroids; probing the farthest lists; unseeded initialisation (irreproducible indexes).
- `recall_at_k` comparing positions (`approx == exact`) instead of sets, or dividing by the number of returned results instead of `k`.

### Interview follow-ups

1. **IVF vs HNSW vs PQ: when do you use which?** IVF-Flat is cheap to build, stores only ids and raw vectors, batches well on GPUs, and has one clear knob (`nprobe`). HNSW (a layered proximity graph) gives the best recall/latency on CPU at high recall, but costs extra memory for neighbour lists (tens to hundreds of bytes per vector), builds slowly and handles deletions poorly. PQ (product quantisation) compresses each vector into `m` sub-codes of 8 bits and scores with lookup tables, which gives 10–100× less memory at some accuracy cost. Billion-scale systems combine them (IVF-PQ, often with HNSW as the coarse quantizer) and **re-rank** the shortlist with full-precision vectors.
2. **Why normalise for cosine, and why do it at index time?** Normalised, cosine ranking equals inner-product ranking and also L2 ranking ($\lVert q - d\rVert^2 = 2 - 2\cos$), so one MIPS or L2 index serves it. CLIP is trained with cosine similarity, so unnormalised dot products would favour large-norm vectors. Normalising once at indexing removes the per-query division.
3. **Which knobs trade recall for latency?** IVF: `nlist` (FAISS suggests on the order of $4\sqrt N$–$16\sqrt N$), `nprobe`, and the k-means training-set size. HNSW: `M`, `efConstruction`, `efSearch`. PQ: `m`, bits per code, and re-ranking depth. You pick the operating point from a recall@k-vs-QPS curve at the latency budget. Recall@10 of 0.9–0.95 is a typical target.
4. **How are CLIP retrieval models evaluated, and how does that "recall@K" differ from ANN recall?** Zero-shot image→text and text→image retrieval on MS-COCO (5k test images, 25k captions) and Flickr30k (1k test images), reporting R@1/5/10: the fraction of queries whose *ground-truth* match appears in the top K. That measures the **model**. ANN recall@k (this notebook) measures the **index**: overlap with exact search over the same embeddings. A perfect index cannot fix a bad embedding model.
5. **How much memory do float32 vectors need compared with PQ codes?** $10^9$ vectors × 768 dims × 4 B = 3.07 TB in float32 (1.5 TB in fp16). PQ with 64-byte codes takes 64 GB, plus 8 GB of int64 ids. That is the difference between a large cluster and a few machines, and it is why billion-scale search compresses vectors.
6. **Why chunk even on a GPU, and how would you batch queries?** A GPU's memory bounds the $(Q, c)$ score block. Chunking keeps peak memory fixed. FAISS-GPU tiles the GEMM and runs a fast k-selection kernel on each tile; fusing the top-k selection into the distance kernel (some libraries, e.g. RAFT/cuVS, do this for small k) avoids writing the scores to memory at all. Queries are batched so each database chunk, loaded once, is reused across many queries, which raises arithmetic intensity. Brute-force GPU search stays competitive up to roughly 10⁷ vectors.